# 🏗️ 10.10 Creating DataFrames

Welcome to **SECTION D: DATAFRAMES (CORE INDUSTRY SKILL)**!

We have officially left the historical context of RDDs behind. From this point forward, you will be learning the exact API that modern Data Engineers use every single day at companies like Netflix, Uber, and Amazon: **The Spark DataFrame API**.

Before we can transform, filter, or aggregate data, we first need to get it into Spark. In this lesson, we will learn how to read raw files (like CSV and JSON) into a structured DataFrame. We will also learn the difference between letting Spark guess your data types versus explicitly telling Spark what your data looks like.

### 🎯 Learning Objectives
* Understand how to load CSV and JSON files into Spark DataFrames.
* Explain what **Schema Inference** is and why it can be dangerous for Big Data.
* Learn how to define an **Explicit Schema** using `StructType`.
* See how defining strict data contracts defines the role of a Data Engineer.

In [ ]:
# First, let's start our SparkSession!
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Creating_DataFrames") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session is running and ready!")

## 1. Reading CSVs and Schema Inference

A **Schema** is the blueprint of your DataFrame. It defines the names of your columns and their data types (e.g., Integer, String, Date).

When you read a file, you can ask Spark to read through the data and *guess* the data types. This is called **Schema Inference**.

Let's create a dummy CSV file and let Spark infer the schema.

In [ ]:
# Step 1: Create a dummy CSV file locally
csv_content = """user_id,username,age,is_active
1,Alice,28,true
2,Bob,35,false
3,Charlie,42,true"""

with open("users.csv", "w") as f:
    f.write(csv_content)

# Step 2: Read the CSV into a DataFrame with inferSchema=True
print("Reading CSV with Schema Inference...")
df_inferred = spark.read.csv(
    "users.csv", 
    header=True,       # Tells Spark the first row contains column names
    inferSchema=True   # Tells Spark to guess the data types
)

# Step 3: Show the data and the guessed schema
df_inferred.show()
print("The Inferred Schema:")
df_inferred.printSchema()

## 2. The Problem with Schema Inference

In the example above, Spark correctly guessed that `user_id` was an Integer, `username` was a String, and `is_active` was a Boolean. It feels like magic! 

So why is `inferSchema=True` considered a bad practice in production Data Engineering?

1. **Performance Cost:** To guess the schema, Spark has to read the *entire* file first just to look at the data types, and then read it *again* to actually process it. If your file is 10 Terabytes, you just doubled your processing time!
2. **Data Corruption:** What if the first 1 million rows of `age` are numbers (like `28`), but row 1,000,001 says `"Twenty-Eight"`? Spark might guess Integer based on a sample, and then crash when it hits the string.

<img src="../assets/Module_10/10_10_01.png" alt="Diagram comparing Schema Inference (Spark using a magnifying glass to look at every row, slowing down) vs Explicit Schema (A blueprint instantly handed to Spark, allowing it to run fast)." width="75%">

## 3. The Professional Way: Explicit Schema

Instead of making Spark guess, a professional Data Engineer explicitly defines the schema upfront. This acts as a strict **Data Contract**.

We do this using `StructType` (which defines the whole table) and `StructField` (which defines a single column).

In [ ]:
# Import the data types we need from PySpark
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType

# Step 1: Define the Explicit Schema
# format: StructField("column_name", DataType(), nullable (True/False))
explicit_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("username", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("is_active", BooleanType(), True)
])

# Step 2: Read the CSV using our custom schema (Notice we removed inferSchema=True)
print("Reading CSV with Explicit Schema...")
df_explicit = spark.read.csv(
    "users.csv", 
    header=True, 
    schema=explicit_schema # Forcing our strict data contract
)

df_explicit.show()
print("The Explicit Schema:")
df_explicit.printSchema()

## 4. Reading JSON Data

CSV files are great for flat, tabular data. But what if you are receiving data from a web API or a mobile app? That data usually comes in **JSON** (JavaScript Object Notation) format.

Spark handles JSON natively. The `spark.read.json()` method works almost exactly like the CSV reader.

<img src="../assets/Module_10/10_10_02.png" alt="A visual showing a CSV document and a JSON document both feeding into a funnel, and coming out the bottom as identical, perfectly structured Spark DataFrames." width="75%">

In [ ]:
# Step 1: Create a dummy JSON file (Notice this is 'JSON Lines' format - one JSON object per line)
json_content = """{"device_id": 101, "os": "iOS", "app_version": "1.4.2"}
{"device_id": 102, "os": "Android", "app_version": "1.4.0"}
{"device_id": 103, "os": "iOS", "app_version": "1.4.2"}"""

with open("devices.json", "w") as f:
    f.write(json_content)

# Step 2: Read the JSON file
# (JSON files contain their own column names, so we don't need 'header=True')
print("Reading JSON data...")
df_json = spark.read.json("devices.json")

df_json.show()
df_json.printSchema()

## 🧑‍💻 The Modern Data Ecosystem: Role Comparison

How do different data professionals interact with data ingestion and schemas?

| Feature | 🏛️ Traditional DBA | 🛠️ Data Engineer (You) | 🧠 Data Scientist |
| :--- | :--- | :--- | :--- |
| **Defining Schemas** | Writes SQL `CREATE TABLE` statements with strict types before any data is inserted. | **Writes PySpark `StructType` to enforce schema on read when pulling data from a Data Lake.** | Generally relies on the DE to clean and structure the data first. |
| **File Formats** | Works entirely inside the database engine. | **Masters reading/writing distributed files (CSV, JSON, Parquet).** | Uses Pandas `read_csv` for local modeling. |
| **Data Quality** | Uses primary keys and constraints to block bad data. | **Defines explicit schemas to prevent messy JSON/CSV data from corrupting the analytical pipelines.** | Spends 80% of their time cleaning data if the DE didn't enforce a schema! |

---

### 📈 Career Progression Roadmap

1. **Junior DE:** Relies heavily on `inferSchema=True`. They build pipelines that work perfectly on small test data, but crash in production when a weird string suddenly appears in an integer column.
2. **Mid-Level DE (Your Goal):** Always uses **Explicit Schemas** (`StructType`). Understands that a schema is a contract that protects the downstream Data Scientists from broken pipelines.
3. **Senior DE:** Builds automated "Schema Evolution" frameworks. When a software developer suddenly adds a new column to the JSON API, the Senior DE's Spark pipeline detects it, alerts the team, and safely updates the schema without crashing the system.

### 🛠️ Skills Matrix
* **Core Object:** Spark DataFrame.
* **Ingestion API:** `spark.read.csv()`, `spark.read.json()`.
* **Schema Tools:** `StructType()`, `StructField()`, `IntegerType()`, `StringType()`.

## 🔑 Key Takeaways

- **DataFrames** are tables of data with rows and named columns, acting like a distributed version of Pandas or an SQL table.
- **Schema Inference (`inferSchema=True`)** asks Spark to guess the data types. It is easy for testing but slow and dangerous for production Big Data.
- **Explicit Schemas (`StructType`)** tell Spark exactly what the data looks like. This skips the guessing phase, making your job faster and immune to unexpected data type changes.
- **`spark.read`** is the gateway to loading data, supporting formats like `.csv()` and `.json()` natively.

## 📝 Knowledge Check Quiz

**Question 1:** Why is using `inferSchema=True` discouraged when reading Petabytes of data in a production environment?
A) Because it converts all data to strings.
B) Because Spark must scan through the massive file an extra time just to guess the data types, doubling the read time.
C) Because it forces Spark to use the slow MapReduce engine.
D) Because inferred schemas cannot be saved to a database.

**Question 2:** Which PySpark object is used to define the overall blueprint/structure of an explicit schema?
A) `SchemaType`
B) `StructField`
C) `DataFrameBlueprint`
D) `StructType`

**Question 3:** When reading a JSON file using `spark.read.json()`, why do you generally not need to specify `header=True` like you do with CSVs?
A) Because JSON files are not allowed to have column names.
B) Because Spark doesn't support headers in JSON.
C) Because JSON is formatted as key-value pairs (e.g., `"username": "Alice"`), meaning the column names (keys) are naturally embedded in every single row.
D) Because JSON data is always unstructured text.

---

*Answers: 1: B, 2: D, 3: C*

In [ ]:
# Clean up our session
import os
spark.stop()

# Optional: Clean up the local dummy files we created
if os.path.exists("users.csv"): os.remove("users.csv")
if os.path.exists("devices.json"): os.remove("devices.json")

print("Spark session closed and temporary files cleaned up.")

### 🚀 What's Next?
We have successfully loaded data into our Spark cluster! 

Now the real fun begins. In the next lesson, **10.11 DataFrame Transformations**, we will learn how to manipulate this data. You will learn the core PySpark commands (`select`, `filter`, `withColumn`) that form the foundation of every Data Engineering pipeline. Let's go!